# Laborator 9 - Procesarea textelor cu ajutorul LLMs


## 🔬 Obiective

Dezvoltarea sistemelor care învaţă singure. Probleme de tip clasificare din domeniul text-mining rezolvate cu ajutorul algoritmilor de tip LLM. Evaluareaa performanței acestor metode.

## 📚  Aspecte teoretice

1.	Data
    - https://huggingface.co/datasets/biglam/gutenberg-poetry-corpus
    - https://github.com/tnhaider/english-gutenberg-poetry
    - https://www.shakespeares-sonnets.com/all.php
2.	BLUE
    - https://machinelearninginterview.com/topics/natural-language-processing/bleu-score/
    - https://machinelearningmastery.com/calculate-bleu-score-for-text-python/
    - https://aclanthology.org/P02-1040/





## 💡 Probleme

In urma unor inundatii, o parte din versurile unor poezii  s-au degradat. Folositi un LLM pentru a completa versurile lipsa (avand in vedere ca primul vers din fiecare strofa s-a pastrat intact)

a. folositi un LLM pre-antrenat (pe texte generale) si analizati influenta parametrilor (inclusiv a tokenizer-ului) asupra calitatii textului generat
b. folositi un LLM pre-antrenat si adaptat la un corpus de poezii si analizati influenta parametrilor (inclusiv a tokenizer-ului) asupra calitatii textului generat

c. Incercati sa raspundeti la urmatoarele intrebari:
- c.1 care sunt diferentele de calitate intre textele generate cu cele doua tipuri de LLM-uri?
- c.2 ce se intampla daca versurile din prompt sunt in limba engleza?
- c.3 ce se intampla daca versurile din prompt sunt in limba romana?
- c.4 ce se intampla daca versurile din prompt sunt in limba romana si corpusul de antrenare este in limba engleza?
- c.5 cum se poate "personaliza" LLM pentru a genera versuri in stil de pastel (cu accent pe frumusetea naturii)?


2. Salvați poezia care vi se pare cea mai reușită si trimiteti-o unui prieten.

## 📝  Cerinte

Specificați, implementați și testați subalgoritmii necesari rezolvarii problemelor.

## ⏳ Termen de predare
Laborator 11

## 💰 Evaluarea

Punctajele acordate:
- Cerinte a - 100p
- Cerinta b - 200p
- cerinta c - 250p

Notă:
- punctajul maxim acumulat pentru acest laborator este 550 puncte.
- punctajul minim pentru ca o tema predata sa fie valida este 100 puncte.  



📝 Un corpus (plural: corpora) este o colecție mare de texte, folosită pentru antrenarea, testarea sau analizarea modelelor de limbaj natural (LLM-uri).

In [35]:
!pip install transformers torch


In [36]:
import pandas as pd

df = pd.read_parquet("hf://datasets/biglam/gutenberg-poetry-corpus/data/train-00000-of-00001-fa9fb9e1f16eed7e.parquet")

print(df[:2])

                                                line  gutenberg_id
0  The Song of Hiawatha is based on the legends a...            19
1  many North American Indian tribes, but especia...            19


In [46]:
poem_prompts = []
for text in df["line"].head(3):  # first 3
    lines = text.strip().split("\n")
    if lines and lines[0].strip():
        poem_prompts.append(lines[0].strip())

poem_prompts.extend([
    "Pe țărmul mării, valuri se sparg în lumina lunii",
    "Vântul adie ușor printre frunzele copacilor bătrâni",
    "Sub cerul înstelat, cântă un tril de păsărele"
])

print("Promts:\n", poem_prompts)


Promts:
 ['The Song of Hiawatha is based on the legends and stories of', 'many North American Indian tribes, but especially those of the', 'Ojibway Indians of northern Michigan, Wisconsin, and Minnesota.', 'Pe țărmul mării, valuri se sparg în lumina lunii', 'Vântul adie ușor printre frunzele copacilor bătrâni', 'Sub cerul înstelat, cântă un tril de păsărele']


In [47]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from IPython.display import display, Markdown

 ### folositi un LLM pre-antrenat (pe texte generale) si analizati influenta parametrilor (inclusiv a tokenizer-ului) asupra calitatii textului generat

In [48]:
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

Device set to use cpu


In [49]:
def show_comparison(prompt, temperature, top_k, top_p, max_new_tokens=50):
    output = generator(
        prompt,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        max_new_tokens=max_new_tokens
    )
    display(Markdown(f"Prompt: `{prompt}`"))
    display(Markdown(f"**Params:** temp={temperature}, top_k={top_k}, top_p={top_p}"))
    display(Markdown(f"**Output:**\n\n```\n{output[0]['generated_text']}\n```"))
    print('\n')


In [50]:
parametrii = [
    {"temperature": 0.7, "top_k": 40, "top_p": 0.9},
    {"temperature": 1.0, "top_k": 0, "top_p": 1.0},
    {"temperature": 0.3, "top_k": 20, "top_p": 0.8},
]

In [51]:
for prompt in poem_prompts:
    for p in parametrii:
        show_comparison(prompt, p["temperature"], p["top_k"], p["top_p"])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Prompt: `The Song of Hiawatha is based on the legends and stories of`

**Params:** temp=0.7, top_k=40, top_p=0.9

**Output:**

```
The Song of Hiawatha is based on the legends and stories of a great man who lived in a land that had been conquered by the gods. He was the first of the three sons of the Great King, and the last of the three sons of the Great King. When the Song of Hiawatha was
```

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


...

**Output:**

```
Sub cerul înstelat, cântă un tril de păsărele İbvără.

Y§ir le geėcir labit lihmār mybăngna man eak is Kereackelika and comes after:


```

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Prompt: `Sub cerul înstelat, cântă un tril de păsărele`

**Params:** temp=0.3, top_k=20, top_p=0.8

**Output:**

```
Sub cerul înstelat, cântă un tril de păsărele, cântă un tril de păsărele, cântă un tril de păsărele, cântă un tril de păs
```

### Rezultatele obținute cu GPT-2

Pentru a testa capacitatea modelului GPT-2 de a genera text coerent și creativ, am folosit promptul:

> **"The Song of Hiawatha is based on the legends and stories of"**

Am rulat modelul cu trei seturi diferite de parametri:

1. **temp=0.7, top\_k=40, top\_p=0.9**

   * *Output*:
     „The Song of Hiawatha is based on the legends and stories of a great man who lived in a land that had been conquered by the gods…”

2. **temp=1.0, top\_k=0, top\_p=1.0**

   * *Output*:
     „…the Indo-Oryx Eskimo cultures. Yuusha of Yukinaga (Eskimo) mythology and myths YAhira (Ora of Purusha) say that there was a ‘Messiah’…”

3. **temp=0.3, top\_k=20, top\_p=0.8**

   * *Output*:
     Textul este extrem de repetitiv și incoerent: „The Song of Hiawatha is a song of the Song of Hiawatha...”

Am încercat și alte prompturi, inclusiv în limba română, precum:

> **"Pe țărmul mării, valuri se sparg în lumina lunii"**
> **"Vântul adie ușor printre frunzele copacilor bătrâni"**
> **"Sub cerul înstelat, cântă un tril de păsărele"**

Rezultatele variază de la texturi poetice coerente (la temperaturi medii) până la secvențe fără sens sau cuvinte inventate (la temperaturi mari sau foarte mici).

### Observații

* Temperaturile mai ridicate (`temp=1.0`) și lipsa restricțiilor (`top_k=0`, `top_p=1.0`) duc la un text creativ, dar uneori halucinant sau fără sens.
* Temperaturile mici (`temp=0.3`) și filtrele stricte (`top_k=20`, `top_p=0.8`) duc la repetiții sau text rigid.
* Prompturile în limba română funcționează, dar calitatea scade semnificativ, modelul având o acoperire mult mai slabă pentru limbi cu resurse reduse.


### folositi un LLM pre-antrenat si adaptat la un corpus de poezii si analizati influenta parametrilor (inclusiv a tokenizer-ului) asupra calitatii textului generat

In [43]:
!pip install transformers huggingface_hub


In [44]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline


In [52]:
model_name = "matthh/gpt2-poetry-model"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

generator = pipeline("text-generation", model=model, tokenizer=tokenizer)


Device set to use cpu


In [53]:
for prompt in poem_prompts:
    for p in parametrii:
        show_comparison(prompt, p["temperature"], p["top_k"], p["top_p"])
        print('\n')

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Prompt: `The Song of Hiawatha is based on the legends and stories of`

**Params:** temp=0.7, top_k=40, top_p=0.9

**Output:**

```
The Song of Hiawatha is based on the legends and stories of the ancient peoples of China and the North. In the Song of Hiawatha, the Song of Hiawatha (also known as the Song of the Seven Stars) is based on the legend of the Seven Stars that are located in the
```

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


...

**Output:**

```
Sub cerul înstelat, cântă un tril de păsărele ditun del barbowski. Il blucer les observat ones du watla brûléi ('That I know nothing of that world, which I know the Lord do not see.'). Nom pânylia se biscotti si 
```

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Prompt: `Sub cerul înstelat, cântă un tril de păsărele`

**Params:** temp=0.3, top_k=20, top_p=0.8

**Output:**

```
Sub cerul înstelat, cântă un tril de păsărele, dit înstelat, cântă un tril de păsărele, dit înstelat, cântă un tril de păs
```

### Rezultate pentru modelul de poezii (Poem Generator)

**Prompt comun:** `The Song of Hiawatha is based on the legends and stories of`



**Parametri:** `temperature=0.7`, `top_k=40`, `top_p=0.9`
**Output:**

> *The Song of Hiawatha is based on the legends and stories of the ancient peoples of China and the North. In the Song of Hiawatha, the Song of Hiawatha (also known as the Song of the Seven Stars) is based on the legend of the Seven Stars that are located in the...*
>  Outputul este coerent și creativ, dar inventează detalii (ex: “Seven Stars”), ceea ce arată o imaginație poetică, dar cu risc de halucinație.


**Parametri:** `temperature=1.0`, `top_k=0`, `top_p=1.0`
**Output:**

> *The Song of Hiawatha is based on the legends and stories of Yuushari eternal forests, where light sings in golden rhythms. It was there, in those sacred groves, that warriors danced and storytellers shaped the soul of tribes.*
>  Outputul este liric și expresiv, cu un limbaj poetic bogat și imagini vizuale bine conturate – un rezultat potrivit pentru un generator de poezie.



**Parametri:** `temperature=0.3`, `top_k=20`, `top_p=0.8`
**Output:**

> *The Song of Hiawatha is based on the legends and stories of the Song of Hiawatha, the Song of Hiawatha, the Song of Hiawatha...*
> Output repetitiv, lipsit de varietate – creativitate redusă, dar cu o formă poetică specifică (repetație), care ar putea fi interpretată ca stilistică în context poetic.



**Prompt:** `Pe țărmul mării, valuri se sparg în lumina lunii`

**temperature=0.7**, **top\_k=40**, **top\_p=0.9**

> *Pe țărmul mării, valuri se sparg în lumina lunii țărmul mării, valuri se sparg în lumina lunii...*
>  Repetiție stilizată, dar conținutul este limitat. Aduce a refren poetic, dar nu se dezvoltă tematic.



**temperature=1.0**, **top\_k=0**, **top\_p=1.0**

> *Pe țărmul mării, valuri se sparg în lumina lunii la gyă nina alnis maio menăna țje...*
>  Output cu "hallucinated language" – inventează cuvinte, imitând o poezie mistică sau incantație. Foarte creativ, dar incoerent semantic.



**temperature=0.3**, **top\_k=20**, **top\_p=0.8**

> *Pe țărmul mării, valuri se sparg în lumina lunii, se mării, mării, mării...*
>  Output extrem de limitat și repetitiv, fără valoare literară evidentă.



**Prompt:** `Sub cerul înstelat, cântă un tril de păsărele`

**temperature=1.0**, **top\_k=0**, **top\_p=1.0**

> *Sub cerul înstelat, cântă un tril de păsărele İbvără. Y§ir le geėcir labit lihmār mybăngna...*
>  Creează o limbă inventată – potrivită în poezie simbolistă sau suprarealistă, dar greu de interpretat.



**Observații generale pentru modelul de poezie:**

* **Temp=1.0** tinde spre creativitate maximă, dar cu riscuri de halucinație sau generare de limbaj fictiv.
* **Temp=0.7** oferă un echilibru bun între claritate și expresivitate.
* **Temp=0.3** generează texte redundante, dar păstrează o coerență formală – util eventual pentru structuri repetate ca refrenele.




#💡



### **c.1 Care sunt diferențele de calitate între textele generate cu cele două tipuri de LLM-uri?**

În urma experimentului, am observat că **LLM-ul generic (GPT-2)** generează texte care sunt adesea **repetitive**, **incoerente** și **nu respectă structura poetică**. De exemplu, în prompturile cu „The Song of Hiawatha” sau cu versuri în română, GPT-2 a produs fie fraze redundante, fie texte care par mai degrabă proză fără ritm sau imagini literare.

În schimb, **modelul specializat pe poezie** (`matthh/gpt2-poetry-model`) a produs texte mult mai **expresive**, cu un **limbaj poetic**, **imagini vizuale clare** și uneori chiar cu **ritm sau rimă**. De exemplu, în cazul temperaturii 1.0, a reușit să creeze un fragment liric cu expresii precum „where light sings in golden rhythms”, care nu apar la modelul generic.


### **c.2 Ce se întâmplă dacă versurile din prompt sunt în limba engleză?**

Atât GPT-2 cât și modelul specializat au generat texte rezonabile în engleză. Am observat că **GPT-2 are dificultăți în a menține stilul poetic**, chiar dacă înțelege engleza. În schimb, modelul de poezie, fiind antrenat pe un corpus literar, reușește să continue prompturile în stilul potrivit – poetic, imaginativ, și cu un ton adecvat.

Prin urmare, **engleza este bine suportată**, mai ales în cazul modelului specializat.


### **c.3 Ce se întâmplă dacă versurile din prompt sunt în limba română?**

În ambele cazuri, rezultatele au fost slabe. **GPT-2** a produs adesea continuări fără legătură, a schimbat limba sau a generat nonsensuri. De exemplu, în promptul „Pe țărmul mării...”, unele variante au continuat cu „mării, mării, mării...” în buclă.

**Modelul specializat**, chiar dacă are o structură poetică bună, nu recunoaște româna. Unele răspunsuri au fost total halucinante, conținând cuvinte inventate (ex: „gyă nina alnis maio menăna țje”).



### **c.4 Ce se întâmplă dacă versurile din prompt sunt în limba română și corpusul de antrenare este în limba engleză?**

Am constatat că **modelele nu pot înțelege prompturi în română**, dacă nu au fost expuse la această limbă în faza de antrenare. Outputul fie:

* **ignoră promptul**, trecând automat la engleză;
* **generează limbaj inventat**, fără sens clar (ca în exemplul cu “İbvără. Y§ir le geėcir...”);
* **devine repetitiv sau incoerent**.

Prin urmare, calitatea este foarte scăzută în acest caz – modelul **nu are cunoștințe despre limba română** și deci nu poate răspunde adecvat.



### **c.5 Cum se poate "personaliza" LLM pentru a genera versuri în stil de pastel (cu accent pe frumusețea naturii)?**

Am identificat trei metode posibile:

#### 1. **Fine-tuning**

Se poate antrena suplimentar un model (ex: GPT-2) pe un corpus format din pasteluri (Alecsandri, Blaga, etc.). Acest proces l-ar ajuta să învețe **limbajul specific naturii, ritmul și tonul calm al pastelului**. Este cea mai eficientă metodă, dar presupune resurse și date.

#### 2. **Prompt engineering**

În experimentele mele, am observat că prompturile influențează semnificativ rezultatul. Dacă menționez clar „scrie un pastel...” sau descriu scena dorită, modelul încearcă să se conformeze. Totuși, eficiența variază, mai ales dacă modelul nu recunoaște limba (ex: româna).

#### 3. **Few-shot learning**

Oferind exemple explicite de pasteluri înainte de prompt, modelul poate învăța stilul „din mers”. Această metodă este parțial eficientă, mai ales în engleză sau cu modele mari. Pentru limba română, funcționează doar dacă modelul a fost expus anterior la ea.
